# Run 기준 전체 불량 재분석

이 노트북은 기존 Timestamp 연결 결과와 Shot별 Run 피처를 사용해 Run 전체·Run 내부 변동폭을 비교합니다. 이전의 개별 센서값 이상치 표를 대체하는 관점을 제공합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

root = Path.cwd()
if not (root / 'reports').exists():
    root = root.parent

shots = pd.read_csv(root / 'reports/tables/run_shot_feature_candidates.csv')
shots['TimeStamp'] = pd.to_datetime(shots['TimeStamp'])
all_bad = pd.read_csv(root / 'reports/tables/all_defect_shot_diagnostics.csv')
all_bad['TimeStamp'] = pd.to_datetime(all_bad['TimeStamp'])
run_summary = pd.read_csv(root / 'reports/tables/run_summary.csv')
assert len(all_bad) == 44

def fnum(value, digits=3):
    return '—' if pd.isna(value) else ('{:.' + str(digits) + 'f}').format(value)

lines = [
    '# 불량 전체: Run 기준으로 다시 본 결과',
    '',
    '한 Shot의 센서 절대값이 아니라 (1) 후보 Run 전체의 수준, (2) 같은 Run의 라벨 정상 Shot 대비 최근 10 Shot 변동폭, (3) Run 내부의 순간 변화로 나누어 읽는다.',
    '',
    'Run은 실제 작업 로그가 아니라 S14 전체 관측에서 제품 전환 또는 30분 넘는 관측 공백을 경계로 나눈 후보 구간이다. 비라벨 Shot은 앞선 10 Shot의 센서 이력을 계산할 때만 사용하고 정상으로 간주하지 않았다.',
    '',
    '## 후보 Run별 라벨 구성',
    '',
    '| 제품 | 후보 Run 시작 | 전체 관측 Shot | 라벨 Shot | 라벨 불량 Shot | 주의 |',
    '|---|---|---:|---:|---:|---|',
]
for run in run_summary.itertuples(index=False):
    note = 'CN7 라벨 있는 후보 Run은 하나뿐' if run.product == 'CN7' else '모든 RG3 후보 Run에 불량 존재'
    lines.append('| {} | {} | {:,} | {} | {} | {} |'.format(
        run.product, run.run_start[:16], int(run.observed_shots),
        int(run.labeled_shots), int(run.bad_shots), note))
lines += [
    '',
    '## 같은 Run 안에서 보인 차이',
    '',
    '| 제품·불량 | 최근 10 Shot 변화/변동폭 | 같은 Run 라벨 정상과 비교 | 판단 |',
    '|---|---|---|---|',
]
cn = shots[shots['product'].eq('CN7')]
cn_pure_gas = cn[cn['Reason'].eq('가스')]
cn_normal = cn[cn['PassOrFail'].eq(0)]
cn_std = cn_pure_gas['Barrel_Temperature_2__past10_std'].median()
cn_norm_std = cn_normal['Barrel_Temperature_2__past10_std'].median()
cn_hopper = cn_pure_gas['Hopper_Temperature__past10_std'].median()
cn_norm_hopper = cn_normal['Hopper_Temperature__past10_std'].median()
lines.append('| CN7 순수 가스 11 Shot | 배럴온도2 과거10샷 표준편차 {} / 호퍼온도 {} | 정상 {} / {} | 온도 자체의 급등보다 직전 온도 흔들림이 후보. 모두 한 사건에 몰림 |'.format(
    fnum(cn_std), fnum(cn_hopper), fnum(cn_norm_std), fnum(cn_norm_hopper)))
cn_mis = cn[cn['Reason'].str.contains('미성형')]
lines.append('| CN7 미성형 3 Shot | 최대 사출속도 과거10샷 중앙값 대비 {}~{} | 정상 샷에서는 이런 연속 급상승 드묾 | 가스 연속 구간 끝에서 속도 상승과 충전시간 감소가 함께 발생 |'.format(
    fnum(cn_mis['Max_Injection_Speed__dev10'].min(), 2),
    fnum(cn_mis['Max_Injection_Speed__dev10'].max(), 2)))
for run_id, g in shots[shots['product'].eq('RG3')].groupby('run_id', sort=False):
    gas = g[g['Reason'].str.contains('가스')]
    normal = g[g['PassOrFail'].eq(0)]
    cushion_gas = gas['Cushion_Position__past10_std'].median()
    cushion_normal = normal['Cushion_Position__past10_std'].median()
    barrel_gas = gas['Barrel_Temperature_5__past10_std'].median()
    barrel_normal = normal['Barrel_Temperature_5__past10_std'].median()
    lines.append('| RG3 가스 {} Shot ({}) | 쿠션위치 과거10샷 표준편차 {} / 배럴온도5 {} | 정상 {} / {} | 쿠션 변동폭은 약간 높지만 차이가 작음; 배럴온도5 방향은 Run마다 다름 |'.format(
        len(gas), run_id[:8], fnum(cushion_gas), fnum(barrel_gas),
        fnum(cushion_normal), fnum(barrel_normal)))
    mis = g[g['Reason'].str.contains('미성형')]
    if len(mis):
        lines.append('| RG3 미성형 {} Shot ({}) | 배럴온도5 과거10샷 표준편차 {} | 정상 {} | 소수 사례; Run마다 같지 않아 보편 규칙 아님 |'.format(
            len(mis), run_id[:8],
            fnum(mis['Barrel_Temperature_5__past10_std'].median()),
            fnum(barrel_normal)))
lines += [
    '| CN7 초기 세팅 5 Shot | 사출속도 38.5~53.3, 충전시간 4.58~8.27 | 10/27 이후 정상 102 Shot 중앙값 55.2, 4.51 | 대회 범위 밖의 한 초기 구간. 이후 정상은 예측 시점에 사용 불가 |',
    '',
    '## 모든 불량 Shot의 Run 맥락',
    '',
    '표의 σ10은 **해당 Shot 직전 10개 관측 Shot의 표준편차**다. 현재·미래 Shot은 포함하지 않는다. 값이 높아도 단독으로 불량이라고 판정할 수 없다. CN7의 10/27 초기불량은 대회용 후보 Run 표에 포함되지 않아 별도 표시한다.',
    '',
    '| 제품 | 시각 | 불량 쪽·유형 | 후보 Run | Run 내부 센서 맥락 |',
    '|---|---|---|---|---|',
]
index = shots.set_index(['product', 'TimeStamp'])
for bad in all_bad.itertuples(index=False):
    if bad.scope == '대회 라벨':
        row = index.loc[(bad.product, bad.TimeStamp)]
        if bad.product == 'CN7':
            context = '배럴2 σ10 {}; 호퍼 σ10 {}; 속도 Δ10 {:+.2f}'.format(
                fnum(row['Barrel_Temperature_2__past10_std']),
                fnum(row['Hopper_Temperature__past10_std']),
                row['Max_Injection_Speed__dev10'])
        else:
            context = '쿠션 σ10 {}; 배럴5 σ10 {}; 배럴2 σ10 {}'.format(
                fnum(row['Cushion_Position__past10_std']),
                fnum(row['Barrel_Temperature_5__past10_std']),
                fnum(row['Barrel_Temperature_2__past10_std']))
        run_label = row['run_id'][:15]
    else:
        context = bad.sensor_observation
        run_label = '10/27 초기 구간(대회 밖)'
    lines.append('| {} | {} | {} | {} | {} |'.format(
        bad.product, bad.TimeStamp.strftime('%m/%d %H:%M:%S'),
        bad.side_reason.replace('|', '·'), run_label, context))
lines += [
    '',
    '## 해석할 때 주의',
    '',
    '- CN7 가스 11 Shot의 온도 변동폭은 같은 Run 정상 중앙값보다 크지만, 모두 한 번의 연속 사건이다. 독립 Run에서도 반복된다는 근거는 없다.',
    '- RG3는 세 후보 Run 모두 불량이 있어 완전히 정상인 Run과 불량 Run을 비교할 수 없다. Run별 작은 차이는 탐색 후보이지 검증된 탐지 규칙이 아니다.',
    '- RH/LH가 같은 Shot의 센서값을 공유해도 불량 라벨은 다를 수 있다. 특히 RG3 불량 25 Shot은 모두 RH에 기록되어 있다.',
    '- 이 표의 Run은 제품 전환·시간 공백으로 추정한 것이며, 설정 변경 로그가 없어 실제 생산 Run과 다를 수 있다.',
]
report = root / 'reports/all_defect_run_context_summary_competition_legacy.md'
report.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('Saved:', report, 'defect rows:', len(all_bad))
print('CN7 pure gas barrel2 std:', fnum(cn_std), 'normal:', fnum(cn_norm_std))
print('CN7 pure gas hopper std:', fnum(cn_hopper), 'normal:', fnum(cn_norm_hopper))